# Shared data on Kaggle: download, splits, pairs (run once for the team)

Settings: **CPU is enough** (no GPU needed), **Internet on**, and a Kaggle secret **`HF_TOKEN`** (a read
token from huggingface.co) attached under Add-ons → Secrets. Without it the download runs anonymously and
will likely hit Hugging Face rate limits.

Output (`/kaggle/working`, kept with the notebook version):

- `AnimeFaceColorization.tar` — the full HF mirror (~8.4 GB) as one uncompressed archive (one file instead of
  ~200k, which Kaggle outputs handle badly). Other notebooks attach this notebook's output and extract it
  (section 5).
- `splits/` — `train/val/test/test_cross.txt` + `cross_ref_pairs.csv` + `shuffled_ref_pairs.csv`.
  **Commit these to the repo** (fairness rule 3: generated once, never ad hoc).
- `previews/` — S / Y / R grids to check the loader by eye; `stats.txt` — dataset counts.

In [ ]:
REPO_URL = "https://github.com/hidrochin/IT5413-_Anime-line-art-colorization_Project.git"
BRANCH = "feat/nhpthinh"   # switch to "main" once the PR is merged
!git clone -q -b $BRANCH $REPO_URL /tmp/repo
%cd /tmp/repo
!git log --oneline -1
!pip install -q -r requirements.txt

## 1. Download the HF mirror
Into `/tmp` (not the output folder); packed into the output as one archive at the end.

In [ ]:
import os
from huggingface_hub import snapshot_download

token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
    print("using the HF_TOKEN secret")
except Exception as e:
    print(f"no HF_TOKEN secret ({type(e).__name__}): anonymous download, may be rate-limited")

DATA = "/tmp/AnimeFaceColorization"
os.environ["ANIME_DATA_ROOT"] = DATA
snapshot_download("ComputerVisionAnimeProject/AnimeFaceColorization", repo_type="dataset", local_dir=DATA,
                  max_workers=32, token=token)
!rm -rf $DATA/.cache
!du -sh $DATA

## 2. What is in the mirror
Group = one folder. `AnimeDiffusion_<n>` holds one drawing plus two deformation-flow copies; the other
folders are moeimouto characters with many different drawings (the source of real cross references).

In [ ]:
from pathlib import Path
from scripts.make_splits import scan_pairs, is_aug

groups = scan_pairs(Path(DATA))
ad = {g: s for g, s in groups.items() if g.startswith("AnimeDiffusion_")}
chars = {g: s for g, s in groups.items() if not g.startswith("AnimeDiffusion_")}
sizes = sorted(len(s) for s in chars.values())
lines = [
    f"pairs: {sum(map(len, groups.values()))} in {len(groups)} groups",
    f"AnimeDiffusion groups: {len(ad)} ({sum(map(len, ad.values()))} images, "
    f"{sum(not is_aug(x) for s in ad.values() for x in s)} originals)",
    f"moeimouto characters: {len(chars)} ({sum(map(len, chars.values()))} images)",
    f"drawings per character: min {sizes[0]}, median {sizes[len(sizes) // 2]}, max {sizes[-1]}",
    f"characters with > 15 drawings (eligible for test_cross): {sum(n > 15 for n in sizes)}",
]
print("\n".join(lines))
Path("/kaggle/working/stats.txt").write_text("\n".join(lines) + "\n")

## 3. Splits and pair files
Defaults: test = 579 AnimeDiffusion originals, val = 1,000, test_cross = 40 held-out characters x 15
drawings (600 sketches, none of these characters in training). Adjust `--cross-chars` / `--cross-per-char`
only if section 2 shows too few eligible characters, and tell the team.

If the official AnimeDiffusion test list has arrived, add `--test-ids <file>`.

In [ ]:
!python -m scripts.make_splits --root $DATA --out splits
!python -m scripts.make_pairs --root $DATA --splits-dir splits
!wc -l splits/*.txt splits/*.csv
!mkdir -p /kaggle/working/splits && cp splits/* /kaggle/working/splits/

## 4. Look at the loader output
Rows: sketch (after the shared clean-up) / target / reference.

In [ ]:
from IPython.display import Image, display
for split, mode in [("train", "train"), ("test", "self_ref"), ("test_cross", "cross_ref"), ("test", "shuffled_ref")]:
    out = f"/kaggle/working/previews/{split}_{mode}.png"
    !python -m scripts.preview_dataset --split $split --mode $mode --n 12 --out $out
    display(Image(out))

## 5. Pack the mirror for other notebooks
In any other notebook: Add Input → this notebook's output, then

```python
!mkdir -p /kaggle/temp && tar -xf /kaggle/input/<this-notebook>/AnimeFaceColorization.tar -C /kaggle/temp
os.environ["ANIME_DATA_ROOT"] = "/kaggle/temp/AnimeFaceColorization"
```

In [ ]:
!tar -cf /kaggle/working/AnimeFaceColorization.tar -C /tmp AnimeFaceColorization
!ls -la /kaggle/working /kaggle/working/splits